# In this notebook we try to diagonalize the fluxonium Hamiltonian (ATS written as fluxonium)

The ATS Hamiltonian reads $$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\phi_\Sigma)\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta) + 2\Delta E_J\sin(\phi_\Sigma)\sin(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta)$$

We can set $\phi_\Sigma = 0$ so that $$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta),$$
we retrieve the Fluxonium Hamiltonian.

Two caveats : 
- experimentally you don't set $\phi_\Sigma = 0$ exactly (few degrees off) and then $\Delta E_J/E_J < 10 \%$
- there is another term $E_{2J}\cos(2\phi_\Sigma)\cos(2\varphi_a (a+a^\dagger) + 2\varphi_b (b+b^\dagger)+2\phi_\Delta)$

In [ ]:
import dynamiqs as dq
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 3.6 * 2 * np.pi
w_b = 6.6 * 2 * np.pi

phi_a = 0.11
phi_b = 0.2

E_J = 26 * 2 * np.pi 
ELp = 7250*  2 * np.pi
E_2J = (E_J**2 + (0.1*E_J)**2)/2/ELp

# drive/pump base params moved out
epsilon_0_base = 0.08
# compute g2 using jax's bessel
g2 = sp.special.jv(1, epsilon_0_base) * E_J * phi_a**2 * phi_b

In [ ]:
n_a = 20
n_b = 10
a = dq.destroy(n_a)
b = dq.destroy(n_b)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b))

In [ ]:
phi_delta_list = np.linspace(0, 2*np.pi, 101)
transitions_list = []

for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = np.linalg.eigh(H)
    evals = np.real(evals)
    evals = np.sort(evals)

    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:10])

transitions_list = np.array(transitions_list)

In [ ]:
for i in range(1, 10):
    plt.plot(phi_delta_list, transitions_list[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency')
plt.show()

## Shifting a few degrees from zero for $\phi_\Sigma$

In [ ]:
phi_delta_list = np.linspace(0, 2*np.pi, 101)
transitions_list = []

phi_sigma = 3*np.pi/180
for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * np.cos(phi_sigma) * dq.cosm(phi_op) + 2 * E_J*0.1 * np.sin(phi_sigma) * dq.sinm(phi_op)
    H = H.to_numpy()

    evals, evecs = np.linalg.eigh(H)
    evals = np.real(evals)
    evals = np.sort(evals)

    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:10])

transitions_list = np.array(transitions_list)

In [ ]:
for i in range(1, 10):
    plt.plot(phi_delta_list, transitions_list[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency')
plt.show()

# Adding the $\cos(2\phi)$-term
$$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\phi_\Sigma)\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta) + 2\Delta E_J\sin(\phi_\Sigma)\sin(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta) + E_{2J}\cos(2\phi_\Sigma)\cos(2\varphi_a (a+a^\dagger) + 2\varphi_b (b+b^\dagger)+2\phi_\Delta),$$
with $E_{2J} = (E_{J_1}^2+E_{J_2})/4\epsilon_L = ((E_J+\Delta E_J)^2+(E_J-\Delta E_J)^2)/4\epsilon_L = (E_J^2+\Delta E_J^2)/2\epsilon_L$

In [ ]:
phi_delta_list = np.linspace(0, 2*np.pi, 101)
transitions_list = []

for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op) + E_2J * dq.cosm(2*phi_op)
    H = H.to_numpy()

    evals, evecs = np.linalg.eigh(H)
    evals = np.real(evals)
    evals = np.sort(evals)

    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:10])

transitions_list = np.array(transitions_list)

In [ ]:
for i in range(1, 10):
    plt.plot(phi_delta_list, transitions_list[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency')
plt.show()

# We now add a third mode in the ATS 
This mode will have high frequency $\omega_c/2\pi=15$ GHz, and a large $\varphi_{zpf}$, let's take $\varphi_{c} = 10\varphi_a$

$$H = \omega_a a^\dagger a + \omega_b b^\dagger b + \omega_c c^\dagger c - 2E_J\cos\bigl[\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\varphi_c (c+c^\dagger)+\phi_\Delta\bigr],$$

In [ ]:
w_c = 15 * 2 * np.pi
phi_c = phi_a*1.2

n_a = 20
n_b = 6
n_c = 5
a = dq.destroy(n_a)
b = dq.destroy(n_b)
c = dq.destroy(n_c)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b), dq.eye(n_c))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()), dq.eye(n_c))
varphi_c_op_tot = dq.tensor(dq.eye(n_a), dq.eye(n_b), phi_c*(c + c.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), dq.eye(n_b),  w_c * dq.number(n_c))

In [ ]:
phi_delta_list = np.linspace(0, np.pi, 30)
transitions_list = []

for phi_delta in phi_delta_list:
    print(phi_delta/2/np.pi)
    phi_op = varphi_a_op_tot + varphi_b_op_tot + varphi_c_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = sp.sparse.linalg.eigsh(H, k=10, which='SM', tol = 0, maxiter=1e6)
    evals = np.real(evals)
    evals = np.sort(evals)
    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:10])

transitions_list = np.array(transitions_list)

In [ ]:
for i in range(1, 10):
    plt.plot(phi_delta_list, transitions_list[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency')
plt.show()